# 🍽️ Restaurant Landscape Analysis — 6th of October City, Egypt

**Source:** Google Maps data scraped via Apify (Google Places Crawler)
**Goal:** Clean a raw, flattened Google Maps export and turn it into a structured,
analysis-ready dataset covering food-serving businesses in 6th of October City.

**Pipeline:** Raw export → column selection → category filtering → missing-value
handling → text standardization → deduplication → SQLite → SQL analysis → dashboard.

**Storage:** All outputs are saved to Google Drive (`project_root` below) so the
work survives Colab runtime restarts, and can be pushed to GitHub / delivered on
Mostaql directly from the same folder.


## 0. Setup — mount Drive and define project paths

Everything in this notebook reads/writes under `project_root`. Running this cell is required before any other cell.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import os

project_root = '/content/drive/MyDrive/restaurant_project'

folders = [
    f'{project_root}/data/raw',
    f'{project_root}/data/cleaned',
    f'{project_root}/data/analytics',
    f'{project_root}/sql',
    f'{project_root}/app',
]
for f in folders:
    os.makedirs(f, exist_ok=True)

print("Project folders ready:")
for f in folders:
    print(" -", f)

## 1. Load the raw data

The Apify actor used was **Google Places Crawler**, run with three overlapping
search terms (`restaurant`, `مطعم`, `food`) over "6th of October City, Egypt",
capped at 50 results per search — 150 rows total. Since `scrapePlaceDetailPage`
was set to `False`, only the search-result-level fields were collected (no price
tier, hours, or menu data).

In [ ]:
# Upload the raw Apify export once — it gets saved permanently into Drive below.
from google.colab import files

uploaded = files.upload()
raw_filename = list(uploaded.keys())[0]
print("Uploaded:", raw_filename)

In [ ]:
import pandas as pd

df_raw = pd.read_excel(raw_filename)
print("Shape:", df_raw.shape)
df_raw.head(3)

In [ ]:
import shutil

# Keep an untouched copy of the raw export in the raw data layer.
raw_path = f'{project_root}/data/raw/restaurants_raw.xlsx'
shutil.copy(raw_filename, raw_path)
print("Raw file archived at:", raw_path)

## 2. Understand the raw data

480 columns come back from the export, but the vast majority are empty —
artifacts of Excel flattening nested JSON arrays (e.g. `additionalInfo/Parking/0/...`)
that only populate when `scrapePlaceDetailPage: true`. The next cell shows which
columns actually carry data.

In [ ]:
non_empty = df_raw.notna().sum().sort_values(ascending=False)
useful_columns = non_empty[non_empty > 0]

print(f"Total columns: {df_raw.shape[1]}")
print(f"Columns with at least one value: {len(useful_columns)}")
useful_columns.head(30)

## 3. Select the useful columns

Only the fields relevant to this project are kept — identity, category,
location, ratings, contact info, and status flags. Everything else (empty
`additionalInfo/*`, `menu`, `price`, `plusCode`, etc.) is dropped.

In [ ]:
useful_cols = [
    'title', 'placeId', 'categoryName', 'categories/0',
    'address', 'city', 'neighborhood', 'street', 'postalCode',
    'location/lat', 'location/lng',
    'totalScore', 'reviewsCount',
    'reviewsDistribution/fiveStar', 'reviewsDistribution/fourStar',
    'reviewsDistribution/threeStar', 'reviewsDistribution/twoStar', 'reviewsDistribution/oneStar',
    'phone', 'website',
    'permanentlyClosed', 'temporarilyClosed', 'wasOpenAtScrapeTime',
    'isAdvertisement', 'searchString', 'rank', 'scrapedAt', 'imagesCount', 'url'
]

df_slim = df_raw[useful_cols].copy()
print(df_slim.shape)

## 4. Filter to food-serving businesses

The `"food"` search term pulled in some non-restaurant results (grocery stores,
a poultry farm, etc.). These are excluded so the dataset stays focused on
places where you'd actually sit down or order food to eat.

In [ ]:
non_food_categories = [
    'Grocery store', 'Supermarket', 'General store', 'Butcher shop',
    'Market', 'Food processing equipment', 'Home help', 'Poultry farm'
]

df_food = df_slim[~df_slim['categoryName'].isin(non_food_categories)].copy()

print(f"Before filter: {df_slim.shape[0]} rows")
print(f"After filter:  {df_food.shape[0]} rows")
df_food['categoryName'].value_counts()

## 5. Handle missing values

Missing data isn't all the same, so it isn't handled with one blanket rule.
Three distinct strategies are applied, one per bucket of fields:

1. **Ratings / review counts** — a missing `totalScore` means "no reviews yet",
   not "unknown score", so it's flagged (`is_rated`) rather than filled with a
   guessed number. A missing `reviewsCount` genuinely means zero, so that *is*
   filled with `0`.
2. **Sparse text fields** (`neighborhood`, `street`, `phone`, `website`) — Google
   simply doesn't have this data for many small businesses. Filled with an
   explicit `'Not provided'` label rather than left as a bare `NaN`.
3. **Status flag** (`wasOpenAtScrapeTime`) — missing means no live-status signal
   was available, labeled `'Unknown'`.

In [ ]:
# Bucket 1 — ratings & review counts
df_food['is_rated'] = df_food['totalScore'].notna()
df_food['reviewsCount'] = df_food['reviewsCount'].fillna(0).astype(int)

dist_cols = [
    'reviewsDistribution/fiveStar', 'reviewsDistribution/fourStar',
    'reviewsDistribution/threeStar', 'reviewsDistribution/twoStar',
    'reviewsDistribution/oneStar'
]
df_food[dist_cols] = df_food[dist_cols].fillna(0).astype(int)

print(df_food['is_rated'].value_counts())

In [ ]:
# Bucket 2 — sparse text fields
text_fields_unknown = ['neighborhood', 'street', 'phone', 'website']
df_food[text_fields_unknown] = df_food[text_fields_unknown].fillna('Not provided')

# postalCode is numeric, so it keeps a nullable integer type instead of a text placeholder
df_food['postalCode'] = df_food['postalCode'].astype('Int64')

print(df_food[text_fields_unknown].apply(lambda c: (c == 'Not provided').sum()))
print("Missing postalCode:", df_food['postalCode'].isna().sum())

In [ ]:
# Bucket 3 — open/closed status flag
df_food['open_status'] = df_food['wasOpenAtScrapeTime'].map({
    True: 'Open',
    False: 'Closed'
}).fillna('Unknown')

df_food['open_status'].value_counts()

## 6. Standardize `city` (non-destructive)

The raw `city` field has 5 variants of the same place (e.g. `'6th of October City (2)'`,
`'First 6th of October'`, two Arabic variants). These are mapped to one canonical
value in a **new** `city_clean` column — the raw `city` column is kept untouched
so the transformation stays auditable.

In [ ]:
city_mapping = {
    '6th of October City (2)': '6th of October City',
    '6th of October City (3)': '6th of October City',
    'First 6th of October':    '6th of October City',
    'قسم أول 6':                '6th of October City',
    'اكتوبر':                   '6th of October City',
}

df_food['city_clean'] = df_food['city'].map(city_mapping).fillna(df_food['city'])

df_food[['city', 'city_clean']].drop_duplicates()

## 7. Deduplicate

`placeId` is Google's own unique identifier per listing, so it's the correct
deduplication key — unlike `title`, which can legitimately repeat across
different branches of the same business.

In [ ]:
dupe_count = df_food['placeId'].duplicated().sum()
print(f"Duplicate placeId rows: {dupe_count}")

df_food = df_food.drop_duplicates(subset=['placeId'], keep='first')
print("Final cleaned shape:", df_food.shape)

## 8. Export the cleaned layer

From this point on, `df_food` is treated as final — no further cleaning, only analysis outputs.

In [ ]:
cleaned_csv = f'{project_root}/data/cleaned/restaurants_clean.csv'
cleaned_parquet = f'{project_root}/data/cleaned/restaurants_clean.parquet'

df_food.to_csv(cleaned_csv, index=False)
df_food.to_parquet(cleaned_parquet, index=False)

print("Saved:")
print(" -", cleaned_csv)
print(" -", cleaned_parquet)
print(df_food.shape)

## 9. Build the SQLite database

A single, well-typed table (`restaurants`) is appropriate here — the data is
flat with no natural one-to-many relationships. `place_id` is set as the
`PRIMARY KEY`, which enforces the same dedup rule as Step 7 at the database
level.

In [ ]:
schema = """
CREATE TABLE IF NOT EXISTS restaurants (
    place_id            TEXT PRIMARY KEY,
    title               TEXT NOT NULL,
    category_name       TEXT NOT NULL,
    category_primary    TEXT,

    address             TEXT,
    city_raw            TEXT,
    city_clean          TEXT,
    neighborhood        TEXT,
    street              TEXT,
    postal_code         INTEGER,
    latitude            REAL,
    longitude           REAL,

    total_score         REAL,
    reviews_count       INTEGER NOT NULL DEFAULT 0,
    is_rated            INTEGER NOT NULL,
    reviews_five_star   INTEGER DEFAULT 0,
    reviews_four_star   INTEGER DEFAULT 0,
    reviews_three_star  INTEGER DEFAULT 0,
    reviews_two_star    INTEGER DEFAULT 0,
    reviews_one_star    INTEGER DEFAULT 0,

    phone               TEXT,
    website             TEXT,

    open_status          TEXT,
    permanently_closed    INTEGER DEFAULT 0,
    temporarily_closed    INTEGER DEFAULT 0,

    search_string        TEXT,
    rank_in_search        INTEGER,
    scraped_at            TEXT,
    images_count           INTEGER DEFAULT 0,
    maps_url               TEXT
);
"""

schema_path = f'{project_root}/sql/schema.sql'
with open(schema_path, 'w', encoding='utf-8') as f:
    f.write(schema)

print("Schema saved to:", schema_path)

In [ ]:
import sqlite3

rename_map = {
    'placeId': 'place_id', 'title': 'title', 'categoryName': 'category_name',
    'categories/0': 'category_primary', 'address': 'address', 'city': 'city_raw',
    'city_clean': 'city_clean', 'neighborhood': 'neighborhood', 'street': 'street',
    'postalCode': 'postal_code', 'location/lat': 'latitude', 'location/lng': 'longitude',
    'totalScore': 'total_score', 'reviewsCount': 'reviews_count', 'is_rated': 'is_rated',
    'reviewsDistribution/fiveStar': 'reviews_five_star',
    'reviewsDistribution/fourStar': 'reviews_four_star',
    'reviewsDistribution/threeStar': 'reviews_three_star',
    'reviewsDistribution/twoStar': 'reviews_two_star',
    'reviewsDistribution/oneStar': 'reviews_one_star',
    'phone': 'phone', 'website': 'website', 'open_status': 'open_status',
    'permanentlyClosed': 'permanently_closed', 'temporarilyClosed': 'temporarily_closed',
    'searchString': 'search_string', 'rank': 'rank_in_search',
    'scrapedAt': 'scraped_at', 'imagesCount': 'images_count', 'url': 'maps_url',
}
df_sql = df_food.rename(columns=rename_map)[list(rename_map.values())]

db_path = f'{project_root}/data/analytics/restaurants.db'
conn = sqlite3.connect(db_path)
conn.executescript(schema)

df_sql.to_sql('restaurants', conn, if_exists='append', index=False)
conn.commit()

check = pd.read_sql('SELECT COUNT(*) AS n FROM restaurants', conn)
print("Rows in database:", check['n'].iloc[0])
print("Database saved at:", db_path)

## 10. Analysis

Queries run against the SQLite database via the open `conn` connection.

In [ ]:
query_rating_distribution = """
SELECT
    CASE
        WHEN total_score >= 4.5 THEN '4.5+'
        WHEN total_score >= 4.0 THEN '4.0-4.49'
        WHEN total_score >= 3.5 THEN '3.5-3.99'
        WHEN total_score >= 3.0 THEN '3.0-3.49'
        WHEN total_score IS NULL THEN 'Unrated'
        ELSE 'Below 3.0'
    END AS score_bucket,
    COUNT(*) AS n_restaurants
FROM restaurants
GROUP BY score_bucket
ORDER BY n_restaurants DESC;
"""
pd.read_sql(query_rating_distribution, conn)

In [ ]:
query_top_reviewed = """
SELECT title, category_name, total_score, reviews_count
FROM restaurants
WHERE is_rated = 1
ORDER BY reviews_count DESC
LIMIT 10;
"""
pd.read_sql(query_top_reviewed, conn)

In [ ]:
query_category_breakdown = """
SELECT
    category_name,
    COUNT(*) AS n_restaurants,
    ROUND(AVG(total_score), 2) AS avg_score,
    SUM(reviews_count) AS total_reviews
FROM restaurants
GROUP BY category_name
HAVING n_restaurants >= 3
ORDER BY n_restaurants DESC;
"""
pd.read_sql(query_category_breakdown, conn)

In [ ]:
query_neighborhood = """
SELECT
    neighborhood,
    COUNT(*) AS n_restaurants,
    ROUND(AVG(total_score), 2) AS avg_score
FROM restaurants
GROUP BY neighborhood
ORDER BY n_restaurants DESC;
"""
pd.read_sql(query_neighborhood, conn)

In [ ]:
# Data quality summary — a legitimate analysis finding in its own right
n = len(df_food)
print(f"Unrated restaurants:   {(~df_food['is_rated']).sum()} / {n} ({(~df_food['is_rated']).mean():.1%})")
print(f"Missing neighborhood:  {(df_food['neighborhood']=='Not provided').sum()} / {n} ({(df_food['neighborhood']=='Not provided').mean():.1%})")
print(f"Missing website:       {(df_food['website']=='Not provided').sum()} / {n} ({(df_food['website']=='Not provided').mean():.1%})")
print(f"Missing phone:         {(df_food['phone']=='Not provided').sum()} / {n} ({(df_food['phone']=='Not provided').mean():.1%})")

## 11. Map visualization

In [ ]:
import plotly.express as px

fig = px.scatter_mapbox(
    df_food, lat='location/lat', lon='location/lng',
    color='totalScore', size='reviewsCount',
    hover_name='title', hover_data=['categoryName', 'reviewsCount'],
    zoom=11, mapbox_style='open-street-map'
)
fig.show()

In [ ]:
conn.close()

## 12. Project structure in Drive

Confirms everything is saved and ready to be pulled down for GitHub / Mostaql delivery.

In [ ]:
for root, dirs, filenames in os.walk(project_root):
    level = root.replace(project_root, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for fn in filenames:
        print(f"{indent}  {fn}")

## Next steps: delivering this project

**On Google Drive:** everything under `restaurant_project/` is already saved
there — `data/raw`, `data/cleaned`, `data/analytics/restaurants.db`, and
`sql/schema.sql`. This is your permanent backup; re-mounting Drive in any
future Colab session gives full access again.

**On GitHub:**
1. Download this notebook (`File → Download → .ipynb`) into a local project folder.
2. Add the dashboard (`app/app.py`) and this notebook to the same folder.
3. Add a `.gitignore` excluding large/generated files if desired (the cleaned
   CSV and `restaurants.db` are small here, so including them is fine and
   makes the repo runnable out of the box).
4. Write a `README.md` covering: problem, pipeline, tech stack, key findings,
   and how to run (`streamlit run app/app.py`).
5. `git init`, `git add .`, `git commit -m "Initial commit"`, push to a new repo.

**On Mostaql:** package the same repo (or a zip of the project folder) as your
portfolio piece, using the README content as your project description —
problem statement, pipeline diagram, and 2-3 key findings from Section 10 make
a strong, honest showcase of real data-cleaning and analysis work.
